# 05 — Temporal Data Alignment and Point-in-Time Controls

> **Retrospective reconstruction notice**
>
> This notebook was prepared retrospectively from available project specifications, phase reports, repository artefacts and conversation records. It is not an original notebook executed during the historical phase.
>
> Evidence labels:
> - **Historically evidenced** — explicitly supported by an available project report, configuration, manifest, source file or recorded execution output.
> - **Reconstructed** — organised or interpreted retrospectively from available evidence.
> - **Not evidenced / not reproducible** — the available evidence does not establish the original execution, exact result or complete decision trail.
>
> This notebook is documentary unless a cell explicitly states otherwise. It does not download data, modify Raw files or claim that illustrative code has been executed.

## 1. Notebook purpose

Document the temporal alignment and point-in-time policies established during Phase 3. This is a retrospective record of the controls and limitations reported in the Phase 3 final assessment, not evidence that a historical notebook was executed.

No data are downloaded, transformed or persisted by this notebook.

## 2. Time conventions

| Setting | Approved policy |
|---|---|
| Primary market calendar | LSE (`XLON`) |
| Operational timezone | `Europe/London` |
| Internal timestamp storage | UTC |
| Trading cut-off | 17:00 `Europe/London` |
| Signal generation | After market close |
| Earliest execution | Next LSE session |

GMT/BST transitions were validated. Exceptional historical LSE closures and calendar-library historical coverage remain documented limitations.

The 17:00 cut-off and next-session execution rule are the recorded project conventions; this notebook does not independently establish the exchange's closing auction mechanics or execution prices.

## 3. Distinct temporal fields

The system must distinguish:

- **Observation date / period:** the date or period to which a value refers.
- **Publication date and time:** when the value became publicly available.
- **Acquisition timestamp:** when the project retrieved the value.
- **Revision date / vintage:** when a value was revised, where applicable.
- **Decision timestamp:** when a forecast or trading decision is made.
- **Execution timestamp:** when a simulated order is assumed to be filled.

These timestamps are not interchangeable. In particular, observation period must not be used as a proxy for publication availability.

## 4. Core point-in-time rules

1. Preserve original observation dates and periods.
2. Never fabricate availability timestamps.
3. Trace acquisition timestamps.
4. Preserve historical vintages where available.
5. Respect source-specific calendars and native frequencies.
6. Do not automatically forward-fill.
7. Do not create synthetic weekend observations.
8. Do not expose historical decisions to future information.
9. Keep forecast targets separate from model features.
10. Make temporal eligibility decisions auditable.
11. Enforce the following availability conditions:

```text
AvailabilityTimestamp <= DecisionTimestamp
AcquisitionTimestamp <= DecisionTimestamp
```

Missing or ambiguous availability timestamps must not silently pass a PIT validation check.

## 5. Market timing policy

For the initial UK equity research:

- Use the LSE calendar as the primary trading calendar.
- Use `Europe/London` as the operational timezone.
- Generate research signals only after the daily market close.
- Do not assume that a close-derived signal can be filled at that same close.
- The earliest permitted simulated execution is the next LSE session.

This is a temporal safeguard, not a complete execution model. Later backtesting must separately model order timing, opening/next-session prices, spread, slippage, fees, liquidity and corporate actions.

## 6. Macroeconomic availability

Macroeconomic observation periods often precede their publication dates. For example, a monthly inflation observation is not automatically available to a model on the last day of the observation month.

Where release timestamps or vintages are incomplete:

- do not assign invented historical release times;
- retain the source observation period;
- use conservative availability only where explicitly documented;
- retain the dataset's assigned PIT status;
- do not claim strict PIT backtesting eligibility for unvalidated series.

The Phase 3 report records a conservative availability policy of 10:00 on the next London business day for the BoE database series. This is a documented conservative policy, not a reconstructed complete historical release calendar.

## 7. Dataset-specific constraints

| Dataset | PIT status | Temporal restriction |
|---|---|---|
| Bank Rate | `PIT_CONDITIONAL` | Historical publication-time limitations; apply the documented conservative BoE database-series availability policy |
| SONIA | `PIT_CONDITIONAL` | Methodology change effective 23 April 2018; apply the documented conservative BoE database-series availability policy |
| Gilt 2Y / 10Y | `PIT_UNVALIDATED` | Historical availability and vintages unverified; not eligible for strict PIT backtesting |
| CPI / Core CPI / CPIH | `PIT_CONDITIONAL` | Multiple frequency representations; incomplete historical release timestamps and vintages |
| Real GDP / GDP Growth | `PIT_UNVALIDATED` | Historical release timestamps and vintages not fully reconstructed; not eligible for strict PIT backtesting |

The Phase 3 report does not assign formal PIT labels to each equity/index dataset in its equity classification table. No equity PIT label is inferred here.

## 8. Frequency and calendar alignment

- Do not treat daily market prices, business-day rates, monthly inflation and quarterly GDP as synchronous observations.
- Do not create values for weekends or market holidays.
- Do not use forward-fill as a default alignment operation.
- Any later as-of join must be based on the information availability timestamp, not simply the observation period.
- Preserve source-specific calendars, release conventions and native frequencies.
- Where multiple frequency representations exist for the same macroeconomic indicator, retain their identity and do not combine them without an explicit methodology.

A future alignment process must be tested for both temporal correctness and economic meaning. This notebook does not implement that process.

## 9. Illustrative PIT validation pattern (not executed)

The following is a conceptual example only. It is not connected to the project's actual schemas and must not be treated as a production validator.

```python
def assert_point_in_time(data, decision_timestamp):
    required_columns = {
        "availability_timestamp",
        "acquisition_timestamp",
    }
    missing_columns = required_columns.difference(data.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    availability = data["availability_timestamp"]
    acquisition = data["acquisition_timestamp"]

    if availability.isna().any():
        raise ValueError("Missing availability timestamp: PIT eligibility cannot be established")
    if acquisition.isna().any():
        raise ValueError("Missing acquisition timestamp: provenance cannot be established")

    if (availability > decision_timestamp).any():
        raise ValueError("Future information detected: availability timestamp")
    if (acquisition > decision_timestamp).any():
        raise ValueError("Future information detected: acquisition timestamp")
```

A production implementation must additionally enforce timezone-aware timestamps, dataset-specific availability policies, vintage semantics, decision-time alignment and explicit handling of incomparable or missing timestamps. The illustrative code has not been executed.

## 10. Evidence classification

- **Historically evidenced:** calendar/timezone conventions, next-session execution rule, PIT rules, dataset classifications and the BoE database-series conservative availability policy are reported in the Phase 3 final report.
- **Reconstructed:** narrative explanation, field definitions and illustrative validation pattern.
- **Not evidenced / not reproducible:** a complete historical release calendar for every macroeconomic observation, all original source-vintage records and original cell-level notebook execution.

The illustrative code is documentation only and must not be represented as an implemented project validator.

## 11. Conclusion and downstream restrictions

Temporal alignment is a prerequisite for credible forecasting research. A dataset can be useful for exploratory analysis while remaining ineligible for strict PIT backtesting.

Future phases must preserve the distinction between observation time, public availability, acquisition time, decision time and execution time. Convenience must never justify relaxing these controls.